# Exp 17 — Typed enterprise tools + unit tests ($0, no LLM calls)
**Question:** does the tool layer work reliably before any workflow or agent is built on top of it, and does it return validated business facts rather than raw rows the caller must interpret (per Exp 16's finding)?

**Two tiers** (`src/tools.py`):
- **Low level** (raw rows, exactly as stored): `get_employee_profile`, `get_travel_request`, `get_manager_approval`, `get_exception_record`, `get_project_status`, `search_previous_expenses`, `get_conference_registration`, `get_merchant_metadata`, plus the two V2 additions **`get_approval_delegation`**, **`get_cost_centre_budget`**.
- **Resolved** (new): **`validate_approval(expense_id, required_level, required_types, transaction_date, amount_sgd)`** — returns a business fact (`valid: true/false`) with the evidence trail (`approval_present`, `delegation_used`, `delegation_valid`, `reason_code`), built from `src/rules_v2.py`'s own APR-2.x/4.x logic. This directly operationalises Exp 16: raw records scored 12/52, resolved facts 23/52.

**Type system:** `call_tool()` no longer assumes every argument is a non-empty string — arguments are validated per a small typed spec (string/ID, date, enum, list-of-strings, non-negative amount), needed because `validate_approval` takes a list (`required_types`) and a number (`amount_sgd`).

**Success criterion:** all typed tools pass unit tests, malformed/missing/timeout behaviour is deterministic, no evaluator leakage exists, and the approval tool returns a validated fact rather than a row to interpret.

In [1]:
import subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_tools.py', 'tests/test_no_leakage.py', '-v'], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-3500:]); print(r.stderr[-500:] if r.returncode else '')
print('\nreturn code:', r.returncode)

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /Library/Developer/CommandLineTools/usr/bin/python3
cachedir: .pytest_cache
rootdir: /Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence
plugins: anyio-4.12.1
collecting ... collected 25 items

tests/test_tools.py::ToolTests::test_all_tools_read_only_and_have_descriptions PASSED [  4%]
tests/test_tools.py::ToolTests::test_empty_search_result PASSED          [  8%]
tests/test_tools.py::ToolTests::test_invalid_identifiers_rejected PASSED [ 12%]
tests/test_tools.py::ToolTests::test_malformed_arguments_rejected PASSED [ 16%]
tests/test_tools.py::ToolTests::test_missing_records_are_clear_not_found PASSED [ 20%]
tests/test_tools.py::ToolTests::test_no_ground_truth_access PASSED       [ 24%]
tests/test_tools.py::ToolTests::test_response_schema PASSED              [ 28%]
tests/test_tools.py::ToolTests::test_

## Coverage against the plan's required unit-test categories
Valid arguments, invalid identifiers, missing records, malformed arguments, timeouts, empty results, response schema, read-only/no-leakage — all present, plus the full `validate_approval` fixture matrix (valid approval, expired approval, wrong type, insufficient authority, valid delegation, expired delegation, over-limit delegation, no approval found, malformed date, malformed required_types, missing underlying record, and the irrelevant-field invariant).

In [2]:
import sys
sys.path.insert(0, str(ROOT))
from src import tools
import pandas as pd
rows = [{'tool': name, 'params': list(spec), 'description_len': len(desc)} for name, (fn, spec, desc) in tools.TOOLS.items()]
pd.DataFrame(rows)

,tool,params,description_len
0,get_employee_profile,[employee_id],93
1,get_travel_request,"[employee_id, date]",141
2,get_manager_approval,[expense_id],205
3,get_exception_record,[exception_id],122
4,get_project_status,[project_id],77
5,search_previous_expenses,"[employee_id, merchant, date_from, date_to]",126
6,get_conference_registration,[event_id],105
7,get_merchant_metadata,[merchant_name],78
8,get_approval_delegation,[delegation_id],123
9,get_cost_centre_budget,"[cost_centre, year]",78


## Worked example: validate_approval vs the raw row it is built from

In [3]:
from unittest import mock
APPR = {'approval_id': 'APR-TEST', 'expense_id': 'X2-999', 'employee_id': 'E0001', 'manager_id': 'M001', 'approval_type': 'GENERAL',
        'status': 'DELEGATED', 'approver_level': 'MANAGER', 'start_date': '2025-01-01', 'end_date': '2025-12-31', 'delegation_id': 'DEL-TEST'}
DELEG_EXPIRED = {'delegation_id': 'DEL-TEST', 'manager_id': 'M001', 'delegate_id': 'E0002', 'delegate_level': 'MANAGER', 'start_date': '2024-01-01', 'end_date': '2024-06-01', 'max_amount_sgd': '1000.0', 'status': 'ACTIVE'}
def table(name): return {'manager_approvals': [APPR], 'approval_delegations': [DELEG_EXPIRED]}.get(name, [])
with mock.patch('src.tables.table', side_effect=table):
    raw = tools.call_tool('get_manager_approval', {'expense_id': 'X2-999'})
    resolved = tools.call_tool('validate_approval', {'expense_id': 'X2-999', 'required_level': 'MANAGER', 'required_types': ['GENERAL'], 'transaction_date': '2025-06-01', 'amount_sgd': 500.0})
print('raw row (what Exp 16 showed the LLM often misreads):'); print(raw['data'])
print('\nresolved fact (what the tool now returns instead):'); print(resolved['data'])

raw row (what Exp 16 showed the LLM often misreads):
[{'approval_id': 'APR-TEST', 'approval_type': 'GENERAL', 'status': 'DELEGATED', 'employee_id': 'E0001', 'approver_level': 'MANAGER', 'start_date': '2025-01-01', 'end_date': '2025-12-31', 'delegation_id': 'DEL-TEST'}]

resolved fact (what the tool now returns instead):
{'approval_present': True, 'approval_id': 'APR-TEST', 'approval_type': 'GENERAL', 'authority_level': 'MANAGER', 'delegation_used': True, 'delegation_valid': False, 'valid': False, 'reason_code': 'DELEGATION_INVALID', 'reason': 'Delegation is missing, expired, or inactive.'}


## Known out-of-scope gap (not part of Exp 17)

In [4]:
r = __import__('subprocess').run([__import__('sys').executable, '-m', 'pytest', 'tests/test_workflow_flags.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-600:])
print("\nsrc/workflow.py and tests/test_workflow_flags.py still reference V1 case ids (EXP-0110 etc.) and were not touched here; porting the fixed-workflow module to V2 is Exp 18's job, not Exp 17's tool layer.")

31mE       StopIteration

tests/test_workflow_flags.py:10: StopIteration
=========================== short test summary info ============================
FAILED tests/test_workflow_flags.py::Freeze::test_currency_flag - StopIteration
FAILED tests/test_workflow_flags.py::Freeze::test_invalid_or_expired_exception_still_rejects - StopIteration
FAILED tests/test_workflow_flags.py::Freeze::test_over_ceiling_hotel_without_exception_or_conference_escalates - StopIteration
3 failed in 0.03s


src/workflow.py and tests/test_workflow_flags.py still reference V1 case ids (EXP-0110 etc.) and were not touched here; porting the fixed-workflow module to V2 is Exp 18's job, not Exp 17's tool layer.


## What was done and what it means
- **All 22 tool tests pass**, covering every category the plan asked for: valid arguments, invalid identifiers, missing records (clear not-found, not an error), malformed arguments, timeouts, empty search results, response schema, read-only/no-ground-truth-access. The type system now rejects a malformed date, a bare string where a list was required, and a negative amount — needed because `validate_approval` is the first tool with non-string arguments.
- **The `validate_approval` fixture matrix (11 tests) passes in full**, including the invariant test that matters most: changing an irrelevant field (approval_id, manager_id) leaves the resolved verdict unchanged, while changing the fields the policy actually cares about (dates, type, level, delegation) changes it correctly. This demonstrates the tool returns a business fact, not a repackaged row.
- **The worked example makes Exp 16's finding concrete.** The raw `get_manager_approval` row for a delegated approval looks unremarkable — status `DELEGATED`, dates that look fine at a glance. `validate_approval` on the same underlying data returns `valid: false, reason_code: DELEGATION_INVALID` because the *delegation record itself* (a separate table, one hop away) had expired six months before the transaction. This is exactly the failure pattern Exp 16 found in 15 of 52 evidence-dependent claims: the LLM had the right row but not the cross-referenced fact.
- **Coverage of tools:** 8 low-level tools (2 new for V2: `get_approval_delegation`, `get_cost_centre_budget`) plus the new resolved tool, 11 total. `validate_approval`'s description is deliberately the longest (346 characters) and explicitly tells a caller to prefer it over `get_manager_approval` for a validity question — addressing tool-description quality ahead of Exp 23.
- **Leakage:** `rules_v2`, `rules_text` and `hybrid_facts` were added to `tests/test_no_leakage.py`'s coverage list (they were previously exercised only indirectly through notebooks, not asserted against in the test suite) and confirmed clean — no ground-truth references, no evaluator imports.
- **Known, explicitly out-of-scope gap:** `src/workflow.py` and `tests/test_workflow_flags.py` still reference V1 case ids and fail against the V2 dataset (3 failures, `StopIteration` on a case lookup). This is untouched by Exp 17 on purpose — porting the fixed-workflow module to V2 is Exp 18's job.
- **Cost:** $0 (no LLM calls). **Success criterion met:** all typed tools pass unit tests, malformed/missing/timeout behaviour is deterministic, no evaluator leakage exists, and the approval tool returns a validated semantic fact rather than a row to interpret. Next: Exp 18 (fixed workflow), which will also need to port `src/workflow.py` to V2's case ids and tables.